# Fraus - Treino do fusor dos tres sinais

Este notebook treina a regressao logistica que FUNDE os tres sinais do Fraus
(texto, emoji, tempo) num score 0-100, e exporta `modelos/fusor.joblib` --
o artefato que o `app` real (`fraus.api.main`) carrega no boot e sem o qual
ele falha alto, de proposito.

**Depende do notebook 01.** Rode `01_treino_bertimbau.ipynb` ANTES: o sinal de
texto aqui vem do BERTimbau ja fine-tunado, lido de
`/content/drive/MyDrive/fraus/modelos/bertimbau-satisfacao`. Sem esse
diretorio no Drive a primeira celula para com erro explicito.

Diferente do notebook 01, este NAO precisa de GPU: sao ~milhares de inferencias
curtas do BERTimbau em CPU e uma regressao logistica do scikit-learn. Com GPU
disponivel a etapa de features fica mais rapida, mas nada quebra sem ela.

Sequencia:

1. montar o Drive e conferir o artefato do notebook 01;
2. clonar o repositorio e instalar o pacote `fraus`;
3. carregar o B2W-Reviews01 e rotular por `recommend_to_a_friend`;
4. costurar as frases em conversas sinteticas (`fraus.ingest.simulador`);
5. extrair as 16 features de cada conversa (`fraus.fusor.montar_features`);
6. treinar o `Fusor`;
7. avaliar em conjunto de teste separado (acuracia e F1-macro);
8. exportar `fusor.joblib` e `importancias.json` para o Drive.

In [ ]:
# 1. Drive montado e artefato do notebook 01 conferido ANTES de qualquer outra coisa.
from google.colab import drive
drive.mount('/content/drive')

import os

DIR_FRAUS = '/content/drive/MyDrive/fraus'
DIR_MODELO_TEXTO = f'{DIR_FRAUS}/modelos/bertimbau-satisfacao'
DIR_SAIDA = f'{DIR_FRAUS}/modelos'

if not os.path.isdir(DIR_MODELO_TEXTO):
    raise FileNotFoundError(
        f'Modelo de texto nao encontrado em {DIR_MODELO_TEXTO}. '
        'Rode notebooks/01_treino_bertimbau.ipynb ate a ultima celula (ela salva '
        'o modelo e o tokenizador nesse caminho do Drive) e volte para ca. '
        'Sem o sinal de texto nao ha o que fundir. Ver docs/treinamento.md.'
    )

# O tokenizador tambem precisa estar la: modelo sem tokenizador nao carrega.
faltando = [
    arquivo for arquivo in ('config.json', 'tokenizer_config.json')
    if not os.path.isfile(f'{DIR_MODELO_TEXTO}/{arquivo}')
]
if faltando:
    raise FileNotFoundError(
        f'{DIR_MODELO_TEXTO} existe mas esta incompleto (faltam: {faltando}). '
        'Rode de novo a celula de export do notebook 01.'
    )

os.makedirs(DIR_SAIDA, exist_ok=True)
print('modelo de texto ok em', DIR_MODELO_TEXTO)

## 2. O pacote `fraus`

A extracao de features NAO e reimplementada aqui: o notebook usa os mesmos
`fraus.sinais.*`, `fraus.fusor.montar_features` e `fraus.ingest.simulador` que
a API usa em producao. Reimplementar seria criar uma segunda definicao de
feature, e um fusor treinado sobre features diferentes das que o servidor monta
pontua errado sem nunca dar erro.

**Caminho escolhido: clone do GitHub.** E o mais simples de repetir (uma celula,
nada de upload manual a cada sessao do Colab) e deixa registrado no proprio
notebook em qual versao do codigo o artefato foi treinado -- o `git rev-parse`
abaixo imprime o commit. Se o repositorio estiver privado ou fora do ar,
substitua a celula por um upload do `.zip` do repositorio e um
`pip install -q -e /content/fraus`.

In [ ]:
# 2. Pacote fraus instalado a partir do repositorio.
REPO = 'https://github.com/joao-ped/Fraus.git'
RAMO = 'feat/motor-e-dashboard'

!rm -rf /content/fraus
!git clone -q --branch {RAMO} {REPO} /content/fraus
!pip install -q -e /content/fraus

import subprocess

commit = subprocess.check_output(
    ['git', '-C', '/content/fraus', 'rev-parse', '--short', 'HEAD']
).decode().strip()
print('fraus instalado no commit', commit)

from fraus.fusor import NOMES_FEATURES

print(len(NOMES_FEATURES), 'features:', NOMES_FEATURES)

## 3. Rotulo: `recommend_to_a_friend`, nao `overall_rating`

A spec (secao 5.1) declara `recommend_to_a_friend` como a **ancora do NPS
inferido**: "sem ele, o NPS da dashboard seria metrica inventada". O campo e a
pergunta do NPS em pessoa -- "voce recomendaria a um amigo?" -- enquanto
`overall_rating` e satisfacao com o produto. O notebook 01 usou a nota porque
treinava um classificador de SENTIMENTO de texto; aqui, onde o alvo e a
categoria de NPS que a dashboard exibe, o rotulo tem que ser a recomendacao.

Mapeamento:

| `recommend_to_a_friend` | classe | indice |
|---|---|---|
| `Yes` | satisfeito | 2 |
| `No` | insatisfeito | 0 |

### A classe neutra com um rotulo binario

`recommend_to_a_friend` tem duas respostas, e as classes do projeto sao tres --
o indice 1 (neutro) ficaria vazio. Isso nao e um detalhe cosmetico: o score do
fusor e `100 * (P(satisfeito) + 0.5 * P(neutro))`, e um fusor de duas classes
faria o termo do neutro sumir e empurraria todo atendimento morno para uma das
pontas, inflando NPS nos dois sentidos.

Havia duas saidas legitimas: (a) compor a classe neutra com
`overall_rating == 3`, ou (b) treinar em duas classes e derivar o neutro por
uma faixa de score.

**Escolhi (a): a classe neutra e composta por `overall_rating == 3`.** Motivo:
o neutro passa a ser um rotulo OBSERVADO no corpus, e nao um limiar arbitrado
por mim depois do treino. Em (b) a fronteira detrator/neutro/promotor viraria
um numero escolhido a mao sobre o score -- exatamente o tipo de constante
inventada que o resto do projeto recusa (a nota e a categoria ja saem de faixas
de NPS publicadas, nao de chute). O custo de (a) e que o neutro vem de outra
pergunta que nao a recomendacao; a mitigacao e explicita: as duas pontas -- que
sao as que definem NPS, ja que neutros nao entram na conta -- continuam
ancoradas em `recommend_to_a_friend`, e o neutro so ocupa o meio.

Para o neutro nao ficar sub-representado (nota 3 e minoria no B2W), as tres
classes sao subamostradas ao tamanho da menor, o mesmo balanceamento do
notebook 01. E as linhas com `overall_rating == 3` sao removidas das duas
pontas antes do sorteio, para uma resenha nao entrar em duas classes ao mesmo
tempo.

In [ ]:
# 3. B2W-Reviews01 rotulado pela recomendacao.
import pandas as pd

URL_B2W = 'https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv'
bruto = pd.read_csv(URL_B2W, delimiter=';')

colunas = ['review_text', 'overall_rating', 'recommend_to_a_friend']
dados = bruto[colunas].dropna().rename(columns={'review_text': 'texto'})
dados['recomenda'] = dados['recommend_to_a_friend'].str.strip().str.lower()
dados = dados[dados['recomenda'].isin(['yes', 'no'])]

INSATISFEITO, NEUTRO, SATISFEITO = 0, 1, 2


def rotular(linha):
    # A classe neutra vem da nota 3 e TEM PRECEDENCIA: a mesma resenha nao pode
    # ser neutra e ponta ao mesmo tempo.
    if linha['overall_rating'] == 3:
        return NEUTRO
    return SATISFEITO if linha['recomenda'] == 'yes' else INSATISFEITO


dados['rotulo'] = dados.apply(rotular, axis=1)
print('antes do balanceamento:')
print(dados['rotulo'].value_counts().sort_index())

menor_classe = dados['rotulo'].value_counts().min()
dados = (
    dados.groupby('rotulo', group_keys=False)
    .apply(lambda g: g.sample(menor_classe, random_state=42))
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
print('\ndepois do balanceamento (', menor_classe, 'por classe):')
print(dados['rotulo'].value_counts().sort_index())

## 4. Conversas sinteticas

Nenhum corpus publico de resenha PT-BR tem timestamps de dialogo, logo nenhum
deles treina o sinal de tempo. `fraus.ingest.simulador.gerar_lote` costura as
frases rotuladas em dialogos cliente-bot, com latencia, escalacao e abandono
amostrados de distribuicoes calibradas por rotulo a partir dos benchmarks de
live chat. E deterministico por semente.

**Limitacao metodologica, a declarar no relatorio:** o sinal de tempo e
treinado em dados sinteticos calibrados por literatura, nao observados.

Treino e teste sao gerados com SEMENTES DIFERENTES e a partir de fatias
disjuntas das frases -- se as mesmas frases aparecessem nos dois lotes, o
teste mediria memorizacao do texto, nao generalizacao.

In [ ]:
# 4. Frases -> conversas sinteticas, com treino e teste disjuntos.
from fraus.ingest.simulador import gerar_lote

QTD_TREINO = 900
QTD_TESTE = 300
# Frases por classe: teto so para o notebook nao passar horas no BERTimbau.
FRASES_POR_CLASSE = 400


def frases_de(quadro, rotulo, inicio, fim):
    textos = quadro[quadro['rotulo'] == rotulo]['texto'].tolist()
    return [t.strip() for t in textos[inicio:fim] if t and t.strip()]


corte = FRASES_POR_CLASSE * 3 // 4  # 75% das frases para treino
frases_treino = {r: frases_de(dados, r, 0, corte) for r in (0, 1, 2)}
frases_teste = {r: frases_de(dados, r, corte, FRASES_POR_CLASSE) for r in (0, 1, 2)}

for nome, tabela in (('treino', frases_treino), ('teste', frases_teste)):
    print(nome, {r: len(v) for r, v in tabela.items()})

lote_treino = gerar_lote(frases_treino, quantidade=QTD_TREINO, semente=20260813)
lote_teste = gerar_lote(frases_teste, quantidade=QTD_TESTE, semente=999)
print('conversas:', len(lote_treino), 'treino /', len(lote_teste), 'teste')
print('exemplo:', lote_treino[0][0].mensagens[0].texto[:80])

In [ ]:
# 5. Os tres sinais de cada conversa, pelo MESMO codigo que a API usa.
from pathlib import Path

from fraus.fusor import montar_features
from fraus.sinais.texto import ClassificadorTexto

classificador = ClassificadorTexto(Path(DIR_MODELO_TEXTO))


def extrair(lote):
    exemplos, rotulos = [], []
    for indice, (conversa, rotulo) in enumerate(lote):
        exemplos.append(montar_features(conversa, classificador))
        rotulos.append(rotulo)
        if (indice + 1) % 100 == 0:
            print(f'  {indice + 1}/{len(lote)} conversas')
    return exemplos, rotulos


print('extraindo features do treino...')
exemplos_treino, rotulos_treino = extrair(lote_treino)
print('extraindo features do teste...')
exemplos_teste, rotulos_teste = extrair(lote_teste)

print('features de uma conversa:')
for nome, valor in exemplos_treino[0].items():
    print(f'  {nome:34s} {valor:.4f}')

In [ ]:
# 6. Treino do fusor.
from fraus.fusor import Fusor

fusor = Fusor()
fusor.treinar(exemplos_treino, rotulos_treino)
print('fusor treinado em', len(exemplos_treino), 'conversas')

In [ ]:
# 7. Avaliacao no conjunto de teste separado.
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, f1_score

# O Fusor expoe `prever` (classe predita), alem de `pontuar` (score 0-100) e
# `importancias`: a API real nunca precisa da classe, so este notebook de
# avaliacao, para medir acuracia e F1.
preditos = [fusor.prever(exemplo) for exemplo in exemplos_teste]

acuracia = accuracy_score(rotulos_teste, preditos)
f1_macro = f1_score(rotulos_teste, preditos, average='macro')
print(f'acuracia : {acuracia:.4f}')
print(f'f1_macro : {f1_macro:.4f}')
print()
print(classification_report(
    rotulos_teste, preditos,
    labels=[0, 1, 2],
    target_names=['insatisfeito', 'neutro', 'satisfeito'],
    digits=4,
))

# O score 0-100 e o que a API grava: vale olhar a media por classe antes de
# exportar. Se as tres medias vierem coladas, o fusor nao separa nada.
scores = np.array([fusor.pontuar(exemplo) for exemplo in exemplos_teste])
verdadeiros = np.array(rotulos_teste)
for rotulo, nome in ((0, 'insatisfeito'), (1, 'neutro'), (2, 'satisfeito')):
    print(f'score medio {nome:13s}: {scores[verdadeiros == rotulo].mean():.1f}')

In [ ]:
# 8. Export: fusor.joblib e importancias.json.
import json

CAMINHO_FUSOR = f'{DIR_SAIDA}/fusor.joblib'
CAMINHO_IMPORTANCIAS = f'{DIR_SAIDA}/importancias.json'

fusor.salvar(CAMINHO_FUSOR)

importancias = fusor.importancias()
with open(CAMINHO_IMPORTANCIAS, 'w', encoding='utf-8') as arquivo:
    json.dump(
        {
            'importancias': importancias,
            'acuracia': float(acuracia),
            'f1_macro': float(f1_macro),
            'conversas_treino': len(exemplos_treino),
            'conversas_teste': len(exemplos_teste),
            'rotulo': 'recommend_to_a_friend (Yes/No) + overall_rating == 3 para o neutro',
            'classes': ['insatisfeito', 'neutro', 'satisfeito'],
        },
        arquivo,
        ensure_ascii=False,
        indent=2,
    )

print('peso absoluto medio por feature (o grafico da apresentacao):')
for nome, peso in sorted(importancias.items(), key=lambda par: -par[1]):
    print(f'  {nome:34s} {peso:.4f}  {"#" * int(peso * 40)}')

print()
print('gravados:')
print(' ', CAMINHO_FUSOR)
print(' ', CAMINHO_IMPORTANCIAS)

# Prova de que o artefato serve para o app: recarrega do disco e pontua.
from fraus.fusor import Fusor as FusorRecarregado

conferencia = FusorRecarregado.carregar(CAMINHO_FUSOR)
print('score da primeira conversa de teste, apos recarregar:',
      round(conferencia.pontuar(exemplos_teste[0]), 2))

## 9. Para onde levar os artefatos

Baixe do Drive e coloque na raiz do repositorio local:

```
modelos/bertimbau-satisfacao/   <- notebook 01
modelos/fusor.joblib            <- este notebook
modelos/importancias.json       <- este notebook (grafico da apresentacao)
```

Com os dois primeiros no lugar, `uv run uvicorn fraus.api.main:app` sobe o app
REAL -- o que carrega BERTimbau e fusor de verdade, em vez do
`scripts/api_demo.py`, que usa motor duble e nao serve para producao.

`modelos/` esta no `.gitignore`: os artefatos nao vao para o git, ficam
versionados apenas pelo Drive.